In [ ]:
import sys
print(sys.executable)


In [ ]:
pip install kagglehub

In [1]:
import kagglehub 
# Download latest version 
path = kagglehub.dataset_download("andresmgs/plantdec") 
print("Path to dataset files:", path)

100%|██████████| 74.1M/74.1M [00:03<00:00, 21.0MB/s]

Extracting files...


Path to dataset files: C:\Users\PC\.cache\kagglehub\datasets\andresmgs\plantdec\versions\7


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\PC\.cache\kagglehub\datasets\abdallahalidev\plantvillage-dataset\versions\3


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("karagwaanntreasure/plant-disease-detection")

print("Path to dataset files:", path)

100%|██████████| 534M/534M [00:19<00:00, 28.1MB/s] 

Extracting files...


Path to dataset files: C:\Users\PC\.cache\kagglehub\datasets\karagwaanntreasure\plant-disease-detection\versions\1


In [ ]:
pip install numpy pillow matplotlib jupyter scikit-learn


### `02_logistic_regression_trraining.ipynb` eski fit fonksiyonu

In [ ]:
def fit(self, X_train, y_train, num_epochs=10, X_val=None, y_val=None, verbose=True):
        """
        Train the model using batch gradient descent.
        X_train: list of feature vectors (list of floats)
        y_train: list of integer labels
        """
        n_samples = len(X_train)

        for epoch in range(1, num_epochs + 1):
            # Initialize gradients
            grad_W = [
                [0.0 for _ in range(self.num_classes)]
                for _ in range(self.num_features)
            ]
            grad_b = [0.0 for _ in range(self.num_classes)]

            total_loss = 0.0

            # One full pass over training data
            for i in range(n_samples):
                x = X_train[i]
                y = y_train[i]

                logits = self._compute_logits(x)
                probs = softmax(logits)

                # Cross-entropy loss: -log p(y)
                p_y = max(probs[y], 1e-15)
                total_loss += -math.log(p_y)

                # Gradient for softmax + cross-entropy
                # dL/dz_k = p_k - 1(k==y)
                dL_dz = [probs[k] for k in range(self.num_classes)]
                dL_dz[y] -= 1.0

                # Update gradient for W and b
                for k in range(self.num_classes):
                    grad_b[k] += dL_dz[k]
                    for j in range(self.num_features):
                        grad_W[j][k] += dL_dz[k] * x[j]

            # Average gradients
            inv_n = 1.0 / n_samples
            for k in range(self.num_classes):
                grad_b[k] *= inv_n
            for j in range(self.num_features):
                for k in range(self.num_classes):
                    grad_W[j][k] *= inv_n

            # Gradient descent step
            lr = self.learning_rate
            for k in range(self.num_classes):
                self.b[k] -= lr * grad_b[k]
            for j in range(self.num_features):
                for k in range(self.num_classes):
                    self.W[j][k] -= lr * grad_W[j][k]

            avg_loss = total_loss / n_samples

            if verbose:
                # Compute training accuracy
                y_pred_train = self.predict(X_train)
                train_acc = accuracy_score(y_train, y_pred_train)

                if X_val is not None and y_val is not None:
                    y_pred_val = self.predict(X_val)
                    val_acc = accuracy_score(y_val, y_pred_val)
                    print(
                        f"Epoch {epoch}/{num_epochs} - "
                        f"loss: {avg_loss:.4f} - "
                        f"train_acc: {train_acc:.4f} - "
                        f"val_acc: {val_acc:.4f}"
                    )
                else:
                    print(
                        f"Epoch {epoch}/{num_epochs} - "
                        f"loss: {avg_loss:.4f} - "
                        f"train_acc: {train_acc:.4f}"
                    )


Eski fit fonksiyonu

In [ ]:
def fit(self, X, y, num_epochs=10, shuffle=True, verbose=True,
            X_val=None, y_val=None):
        n_samples = len(X)

        for epoch in range(num_epochs):
            indices = list(range(n_samples))
            if shuffle:
                random.shuffle(indices)

            total_loss = 0.0
            correct = 0

            progress_step = max(1, n_samples // 10)

            for step, idx in enumerate(indices):
                x = X[idx]
                true_class = y[idx]

                logits = self._compute_logits(x)
                probs = softmax(logits)
                loss = cross_entropy_loss(probs, true_class)
                total_loss += loss

                pred_class = 0
                best_prob = probs[0]
                for k in range(1, self.num_classes):
                    if probs[k] > best_prob:
                        best_prob = probs[k]
                        pred_class = k
                if pred_class == true_class:
                    correct += 1

                for k in range(self.num_classes):
                    if k == true_class:
                        error_k = probs[k] - 1.0
                    else:
                        error_k = probs[k]

                    for j in range(self.num_features):
                        grad_w_jk = error_k * x[j]
                        self.W[j][k] -= self.learning_rate * grad_w_jk

                    grad_b_k = error_k
                    self.b[k] -= self.learning_rate * grad_b_k

                if verbose and (step + 1) % progress_step == 0:
                    pct = (step + 1) / n_samples * 100.0
                    print(
                        f"Epoch {epoch + 1}/{num_epochs} - "
                        f"{pct:5.1f}% completed",
                        end="\r",
                        flush=True
                    )

            avg_loss = total_loss / n_samples
            train_acc = correct / n_samples

            val_acc = None
            if X_val is not None and y_val is not None and len(X_val) > 0:
                correct_val = 0
                for x_val, y_true_val in zip(X_val, y_val):
                    y_pred_val = self.predict_one(x_val)
                    if y_pred_val == y_true_val:
                        correct_val += 1
                val_acc = correct_val / len(X_val)

            if verbose:
                if val_acc is not None:
                    print(
                        f"Epoch {epoch + 1}/{num_epochs} - "
                        f"loss: {avg_loss:.4f} - "
                        f"train_acc: {train_acc:.4f} - "
                        f"val_acc: {val_acc:.4f}          "
                    )
                else:
                    print(
                        f"Epoch {epoch + 1}/{num_epochs} - "
                        f"loss: {avg_loss:.4f} - "
                        f"train_acc: {train_acc:.4f}          "
                    )